In [2]:
## Imports

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter

from langchain_text_splitters import RecursiveCharacterTextSplitter


c:\Users\reeuwi006\Documents\Thesis\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
df_vh = pd.read_csv(
    r'Data\vakantieverhuur.csv', 
    sep=';', 
    encoding='cp1252'
)

def map_subtype(subject):
    if 'verhuurvergunning' in str(subject).lower(): return 'Rental Permit'
    if 'bed&breakfast' in str(subject).lower(): return 'B&B'
    if 'verbodsgebieden' in str(subject).lower(): return 'Prohibited Areas'
    return 'Other'


def map_description(subject):
    if 'bezwaar' in str(subject).lower(): return 'Decision on Objection'
    if 'advies' in str(subject).lower(): return 'Advice'
    if 'voordracht' in str(subject).lower(): return 'Voordracht'
    if '' in str(subject).lower(): return 'Empty'
    return 'Other'

df_vh['Subtype'] = df_vh['Onderwerp'].apply(map_subtype)
df_vh['Legal_Text'] = df_vh['geanonimiseerd_doc_inhoud']
df_vh['Description'] = df_vh['Omschrijving'].apply(map_description)


print(f"Total cases in dataset: {len(df_vh)}")
print("Number of cases per Subtype (Workload Overview):")
print(df_vh['Subtype'].value_counts())


print("Types of letters:")
print(df_vh['Description'].value_counts())

Total cases in dataset: 329
Number of cases per Subtype (Workload Overview):
Subtype
Rental Permit       192
B&B                 134
Prohibited Areas      2
Other                 1
Name: count, dtype: int64
Types of letters:
Description
Empty                    272
Decision on Objection     31
Advice                    24
Voordracht                 2
Name: count, dtype: int64


In [7]:
df_wrv = pd.read_csv(
    r'Data\WRV_bezwaren_.csv', 
    sep=';', 
    encoding='cp1252'
)

print(f"Total cases in dataset: {len(df_wrv)}")

def subtype(subject):
    subject_lower = str(subject).lower()
    if 'afbouw' in subject_lower and 'alle punten' in subject_lower:
        return 'Reduction of all points'
    if 'start-' in subject_lower and 'situatiepunten' in subject_lower:
        return 'Starting and situation points'
    if 'situatiepunten' in  subject_lower: 
        return 'Situation points'
    if 'startpunten' in  subject_lower: 
        return 'Starting points'
    if 'zoekpunten' in  subject_lower: 
        return 'Search points'
    return 'Other'

df_wrv['Subtype'] = df_wrv['Onderwerp'].apply(subtype)
df_wrv['Legal_Text'] = df_wrv['geanonimiseerd_doc_inhoud']
print("Number of cases per Subtype (Workload Overview):")
print(df_wrv['Subtype'].value_counts())
 

Total cases in dataset: 929
Number of cases per Subtype (Workload Overview):
Subtype
Search points                    586
Situation points                 263
Starting points                   53
Reduction of all points           24
Starting and situation points      3
Name: count, dtype: int64


## Chunking

In [22]:
df = df_wrv
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

df['chunks'] = df['Legal_Text'].apply(lambda x: splitter.split_text(str(x)))

# Pak het eerste document en de bijbehorende chunks
sample_chunks = df_wrv['chunks'].iloc[0]

print(f"--- Inspectie van Document 1 ---")
for i, chunk in enumerate(sample_chunks):
    print(f"\n[Chunk {i+1}] (Lengte: {len(chunk)} characters):")
    print(chunk)
    print("-" * 30)

--- Inspectie van Document 1 ---

[Chunk 1] (Lengte: 665 characters):
Op 1 oktober 2025 maakte u bezwaar tegen twee besluiten van 1 oktober 2025, waarbij
automatisch twee zoekpunten bij WoningNet zijn afgebouwd. De reden hiervoor is dat u heeft
aangegeven geen interesse te hebben in de woningen 7 Amstelkwartier Studio Jongerenwoning
Type B4 en 5 Amstelkwartier Jongerenwoning Type B3 te Amsterdam, of dat u niet (op tijd) uw
reactie heeft doorgegeven na de bezichtiging.

beslissing op bezwaar
Wij verklaren uw bezwaarschrift gegrond. De bestreden besluiten worden herroepen. Dit betekent
dat de besluiten om uw punten met twee zoekpunten af te bouwen, ongedaan worden gemaakt.
Uw twee zoekpunten worden in het systeem weer hersteld.
------------------------------

[Chunk 2] (Lengte: 489 characters):
Bezwaren
U hebt de volgende bezwaargronden aangevoerd.
-  U geeft aan dat u op het moment van het aanbod 28 jaar oud was en daardoor geen recht
had op deze woningen. U hebt telefonisch contact gez